In [1]:
import pandas as pd
import numpy as np

from collections import Counter
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG
# =========================
DATA_PATH = "../data/raw/development.csv"
N_SPLITS = 5
RANDOM_STATE = 42
USE_ONLY_FIRST_FOLD = True

APPLY_RULE_IF_FOUND = True
RULE_PRIORITY = "best_purity_then_freq"
MIN_RULE_SUPPORT = 30

C_VALUES = [1.5, 2.0, 3.0]   # <<< SOLO QUESTO CAMBIA

# =========================
# LOAD + TIMESTAMP DROP
# =========================
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df = df[df["timestamp"].notna()].reset_index(drop=True)

print("Samples after timestamp drop:", len(df))

# =========================
# BASIC FIXES
# =========================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)

# =========================
# TEXT
# =========================
df["text"] = (df["title"] + " " + df["article"]).str.lower()

# =========================
# NUMERIC FEATURES
# =========================
df["n_tokens"] = df["article"].str.split().str.len()
df["title_len"] = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

df["year"]  = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month
df["dow"]   = df["timestamp"].dt.dayofweek

# =========================
# X / y
# =========================
X = df[
    ["source", "text",
     "n_tokens", "title_len", "article_len",
     "title_ratio", "year", "month", "dow"]
]
y = df["label"].astype(int)

# =========================
# RULE TOKENIZATION
# =========================
def tokenize_for_rules(text):
    return text.split()

# =========================
# RULE MINING (TRAIN ONLY)
# =========================
def mine_pure_rules(texts, labels):
    from collections import defaultdict
    counts = defaultdict(lambda: Counter())

    for txt, y in zip(texts, labels):
        for tok in set(tokenize_for_rules(txt)):
            counts[tok][y] += 1

    rule_token_to_class = {}
    rule_meta = {}

    for tok, c in counts.items():
        total = sum(c.values())
        if total < MIN_RULE_SUPPORT:
            continue

        best_class, best_freq = c.most_common(1)[0]
        purity = best_freq / total

        if purity >= 0.90:
            rule_token_to_class[tok] = best_class
            rule_meta[tok] = (purity, total)

    return rule_token_to_class, rule_meta

# =========================
# APPLY RULES
# =========================
def apply_rules(texts, rule_token_to_class, rule_meta):
    rule_pred = np.full(len(texts), -1, dtype=int)
    matched_token = [None] * len(texts)

    for i, txt in enumerate(texts):
        toks = set(tokenize_for_rules(txt))
        hits = [t for t in toks if t in rule_token_to_class]
        if not hits:
            continue

        if RULE_PRIORITY == "best_purity_then_freq":
            hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
        else:
            hits.sort(key=lambda t: rule_meta[t][1], reverse=True)

        best = hits[0]
        rule_pred[i] = int(rule_token_to_class[best])
        matched_token[i] = best

    return rule_pred, matched_token

# =========================
# MODEL FACTORY (C VARIABILE)
# =========================
def make_model(C_VALUE):
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1,2),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),

            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3,5),
                min_df=3,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),

            ("num", StandardScaler(), [
                "n_tokens", "title_len", "article_len",
                "title_ratio", "year", "month", "dow"
            ])
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])

# =========================
# RUN (1 FOLD, MULTI-C)
# =========================
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):

    print(f"\n===== FOLD {fold_id} =====")

    X_tr, y_tr = X.iloc[tr], y.iloc[tr]
    X_te, y_te = X.iloc[te], y.iloc[te]

    # ---- Mine rules on TRAIN
    rule_token_to_class, rule_meta = mine_pure_rules(
        X_tr["text"], y_tr
    )
    print("Mined rules:", len(rule_token_to_class))

    for C_VALUE in C_VALUES:
        print(f"\n--- LogisticRegression C = {C_VALUE} ---")

        model = make_model(C_VALUE)
        model.fit(X_tr, y_tr)

        model_pred = model.predict(X_te)

        rule_pred, matched_token = apply_rules(
            X_te["text"], rule_token_to_class, rule_meta
        )

        final_pred = model_pred.copy()
        mask = rule_pred != -1
        final_pred[mask] = rule_pred[mask]

        print("Macro F1:",
            f1_score(y_te, final_pred, average="macro"))

        print("Rule coverage:",
            f"{mask.mean():.3f}")

    if USE_ONLY_FIRST_FOLD:
        break


Samples after timestamp drop: 52247

===== FOLD 1 =====
Mined rules: 115

--- LogisticRegression C = 1.5 ---
Macro F1: 0.7503053925700244
Rule coverage: 0.106

--- LogisticRegression C = 2.0 ---
Macro F1: 0.747514418020134
Rule coverage: 0.106

--- LogisticRegression C = 3.0 ---
Macro F1: 0.7442488838301059
Rule coverage: 0.106
